# 3. Modelling, validation and applicability

The part everyone skips to, and the part where most published QSAR models
go wrong — not in the fitting, but in the evaluation.

**Covers:** `qsarkit.models`, `qsarkit.model_selection`, `qsarkit.validation`,
`qsarkit.metrics`, `qsarkit.feature_selection`, `qsarkit.applicability`,
`qsarkit.uncertainty`

In [1]:
import numpy as np
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
np.set_printoptions(precision=3, suppress=True)

SMILES = [
    "OC(=O)c1ccccc1", "OC(=O)c1ccc(C)cc1", "OC(=O)c1ccc(Cl)cc1",
    "OC(=O)c1ccc(Br)cc1", "OC(=O)c1ccc(OC)cc1", "OC(=O)c1ccc(N)cc1",
    "CC(=O)Nc1ccccc1", "CC(=O)Nc1ccc(C)cc1", "CC(=O)Nc1ccc(Cl)cc1",
    "CC(=O)Nc1ccc(F)cc1", "CC(=O)Nc1ccc(OC)cc1", "CC(=O)Nc1ccc(O)cc1",
    "NC(=O)c1ccncc1", "NC(=O)c1ccc(C)nc1", "NC(=O)c1ccc(Cl)nc1",
    "NC(=O)c1ccc(OC)nc1", "CNC(=O)c1ccncc1", "CCNC(=O)c1ccncc1",
    "c1ccc2[nH]cnc2c1", "Cc1ccc2[nH]cnc2c1", "Clc1ccc2[nH]cnc2c1",
    "COc1ccc2[nH]cnc2c1", "Cn1cnc2ccccc21", "CCn1cnc2ccccc21",
]
Y = np.array([
    5.10, 5.35, 7.80, 5.40, 5.05, 4.90,
    6.20, 6.45, 6.70, 6.55, 6.10, 6.05,
    7.10, 7.35, 7.55, 7.20, 7.05, 6.95,
    8.00, 8.25, 8.45, 8.10, 7.90, 7.85,
])
mols = [Chem.MolFromSmiles(s) for s in SMILES]

from qsarkit.representation import MorganFingerprint
X = MorganFingerprint(radius=2, n_bits=512).transform(mols)
X.shape, Y.shape

((24, 512), (24,))

## The split is the experiment

A random split of a QSAR dataset measures interpolation. Public datasets are
dense with near-duplicate analogues, so random assignment scatters a
congeneric series across both sides and scores the model on compounds whose
close relatives it has memorized.

Let us measure how much that flatters us.

In [2]:
from qsarkit.model_selection import RandomSplitter, ScaffoldSplitter
from qsarkit.models import QSARRegressor
from qsarkit.metrics import q2_f1, rmse

def fit_and_score(train, test, label):
    model = QSARRegressor("rf", random_state=0).fit(X[train], Y[train])
    pred = model.predict(X[test])
    print(f"{label:18} n_test={len(test):2}  "
          f"RMSE={rmse(Y[test], pred):.3f}  Q2F1={q2_f1(Y[test], pred, Y[train]):.3f}")
    return pred

rnd_train, rnd_test = next(RandomSplitter(test_size=0.25, random_state=0).split(X, Y))
sca_train, sca_test = next(ScaffoldSplitter(test_size=0.25).split_mols(mols, Y))

_ = fit_and_score(rnd_train, rnd_test, "random split")
_ = fit_and_score(sca_train, sca_test, "scaffold split")

random split       n_test= 6  RMSE=0.475  Q2F1=0.823
scaffold split     n_test= 6  RMSE=0.769  Q2F1=-0.889


That gap is the size of the illusion. Both numbers are honestly computed;
only one of them answers the question "will this work on new chemistry".

The scaffold split guarantees no scaffold appears on both sides:

In [3]:
from qsarkit.chemspace import bemis_murcko_smiles

train_sc = {bemis_murcko_smiles(mols[i]) for i in sca_train}
test_sc = {bemis_murcko_smiles(mols[i]) for i in sca_test}
print("training scaffolds:", train_sc)
print("test scaffolds:    ", test_sc)
print("overlap:           ", train_sc & test_sc)

training scaffolds: {'c1ccc2[nH]cnc2c1', 'c1ccccc1'}
test scaffolds:     {'c1ccncc1'}
overlap:            set()


qsarkit ships several splitters, each answering a different question.

In [4]:
from qsarkit.model_selection import (
    ButinaClusterSplitter, KennardStoneSplitter, MaxMinSplitter,
    SphereExclusionSplitter)

for label, splitter, use_mols in [
    ("random", RandomSplitter(test_size=0.25, random_state=0), False),
    ("scaffold", ScaffoldSplitter(test_size=0.25), True),
    ("butina cluster", ButinaClusterSplitter(test_size=0.25), True),
    ("sphere exclusion", SphereExclusionSplitter(test_size=0.25), True),
    ("kennard-stone", KennardStoneSplitter(test_size=0.25), False),
    ("maxmin", MaxMinSplitter(test_size=0.25, random_state=0), False),
]:
    tr, te = (next(splitter.split_mols(mols, Y)) if use_mols
              else next(splitter.split(X, Y)))
    fit_and_score(tr, te, label)

random             n_test= 6  RMSE=0.475  Q2F1=0.823


scaffold           n_test= 6  RMSE=0.769  Q2F1=-0.889


butina cluster     n_test= 6  RMSE=1.683  Q2F1=0.047
sphere exclusion   n_test= 6  RMSE=1.683  Q2F1=0.047


kennard-stone      n_test= 6  RMSE=1.170  Q2F1=0.471
maxmin             n_test= 6  RMSE=0.771  Q2F1=0.634


## Comparing models

`QSARRegressor` dispatches on a name, so comparing backends is a one-word
change. Scored on the scaffold split, which is the honest one.

In [5]:
for name in ("rf", "svm", "gbm", "knn", "pls", "ridge", "gp"):
    model = QSARRegressor(name, random_state=0).fit(X[sca_train], Y[sca_train])
    pred = model.predict(X[sca_test])
    print(f"{name:6} RMSE={rmse(Y[sca_test], pred):.3f}  "
          f"Q2F1={q2_f1(Y[sca_test], pred, Y[sca_train]):+.3f}")

rf     RMSE=0.769  Q2F1=-0.889
svm    RMSE=0.650  Q2F1=-0.352
gbm    RMSE=0.559  Q2F1=+0.000
knn    RMSE=1.306  Q2F1=-4.453
pls    RMSE=0.810  Q2F1=-1.099
ridge  RMSE=0.735  Q2F1=-0.729
gp     RMSE=0.677  Q2F1=-0.464


Negative Q² means the model is worse than predicting the training mean. On
a genuinely hard split with 18 training compounds, that is the expected and
honest result — and it is what a random split would have hidden.

### Any estimator, not just the built-in menu

`name` also accepts anything following the scikit-learn `fit`/`predict`
protocol — XGBoost, LightGBM, CatBoost, or your own wrapper.

In [6]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge

# an instance, used as configured (and cloned, never mutated)
model = QSARRegressor(Ridge(alpha=2.0)).fit(X[sca_train], Y[sca_train])
print("instance: ", type(model.estimator_).__name__, model.estimator_.alpha)

# a class, built from model_args / model_params
model = QSARRegressor(ExtraTreesRegressor,
                      model_params={"n_estimators": 50, "random_state": 0})
model.fit(X[sca_train], Y[sca_train])
print("class:    ", type(model.estimator_).__name__)

# fit_params reaches arguments the constructor does not take
weights = np.linspace(0.5, 1.5, len(sca_train))
model = QSARRegressor(Ridge, fit_params={"sample_weight": weights})
print("fit_params:", model.fit(X[sca_train], Y[sca_train]).predict(X[sca_test]).shape)

instance:  Ridge 2.0
class:     ExtraTreesRegressor
fit_params: (6,)


## Cross-validation

A single split is one sample of a noisy quantity. Cross-validation averages
over several.

In [7]:
from qsarkit.validation import CrossValidator

for method, kwargs in [("kfold", {"n_splits": 5}),
                       ("repeated_kfold", {"n_splits": 5, "n_repeats": 3}),
                       ("loo", {})]:
    report = CrossValidator(method=method, random_state=0, **kwargs).evaluate(
        QSARRegressor("rf", random_state=0), X, Y)
    print(f"{method:16} splits={report['n_splits']:3}  "
          f"Q2={report['q2']:.3f}  RMSE={report['rmse_cv']:.3f}")

kfold            splits=  5  Q2=0.686  RMSE=0.602


repeated_kfold   splits= 15  Q2=0.639  RMSE=0.646


loo              splits= 24  Q2=0.603  RMSE=0.677


`n_splits` is reported from the splitter, not the constructor argument —
leave-one-out on 24 compounds is 24 splits, and a report claiming "5" would
be wrong.

## Feature selection, done correctly

Selecting features using all the labels and *then* splitting is selection
bias: the choice has already seen the test set. Here is that error, measured.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from qsarkit.feature_selection import MutualInformationSelector, VarianceFilter

# WRONG: select on everything, then cross-validate
X_leaky = MutualInformationSelector(k=20).fit_transform(X, Y)
leaky = cross_val_score(QSARRegressor("rf", random_state=0), X_leaky, Y,
                        cv=5, scoring="r2").mean()

# RIGHT: selection inside the pipeline, so it refits on each training fold
honest_pipe = Pipeline([
    ("select", MutualInformationSelector(k=20)),
    ("model", QSARRegressor("rf", random_state=0)),
])
honest = cross_val_score(honest_pipe, X, Y, cv=5, scoring="r2").mean()

print(f"selection outside the loop (optimistic): {leaky:+.3f}")
print(f"selection inside the loop (honest):      {honest:+.3f}")

selection outside the loop (optimistic): +0.394
selection inside the loop (honest):      +0.252


The unsupervised filters are safe either way, because they never look at
`y` — and on fingerprints they are nearly free.

In [9]:
print("before:", X.shape)
print("after variance filter:", VarianceFilter().fit_transform(X).shape)

before: (24, 512)
after variance filter: (24, 121)


## Hyperparameters without lying to yourself

Tuning on the same folds you report scores from leaks the test set into the
choice of hyperparameters. `NestedCV` keeps selection and assessment apart.

In [10]:
from qsarkit.model_selection import NestedCV

nested = NestedCV(
    QSARRegressor("ridge"),
    {"model_params": [{"alpha": 0.01}, {"alpha": 0.1}, {"alpha": 1.0}, {"alpha": 10.0}]},
    inner_cv=3, outer_cv=5,
)
result = nested.run(X, Y)
print(f"nested CV score: {result['mean_score']:.3f} +/- {result['std_score']:.3f}")
print("alpha chosen per outer fold:",
      [p["model_params"]["alpha"] for p in result["best_params"]])

nested CV score: 0.477 +/- 0.492
alpha chosen per outer fold: [10.0, 10.0, 10.0, 10.0, 0.1]


The outer folds disagree about the best alpha. That disagreement is itself
the finding: the choice is not well determined by this much data, so a
single "best" hyperparameter reported from one grid search would be noise.

## The OECD validation metrics

R² alone does not establish that a model predicts. The Golbraikh–Tropsha
criteria are five conditions a predictive QSAR model must satisfy at once.

In [11]:
from qsarkit.metrics import golbraikh_tropsha_criteria, qsar_regression_report

model = QSARRegressor("rf", random_state=0).fit(X[rnd_train], Y[rnd_train])
pred = model.predict(X[rnd_test])

report = qsar_regression_report(Y[rnd_test], pred)
for key in ("r2", "rmse", "mae", "ccc", "q2_f2"):
    print(f"{key:8} {report[key]:+.3f}")

r2       +0.821
rmse     +0.475
mae      +0.455
ccc      +0.871
q2_f2    +0.821


In [12]:
gt = golbraikh_tropsha_criteria(Y[rnd_test], pred, q2=0.61)
for key in sorted(k for k in gt if k.startswith("criterion")):
    print(f"{key:22} {gt[key]}")
print(f"{'PASSED':22} {gt['passed']}")

criterion_1_q2         True
criterion_2_r2         True
criterion_3_r0         False
criterion_4_slope      True
criterion_5_delta_r0   True
PASSED                 False


## Applicability domain

A prediction outside the domain is not *wrong* — it is unsupported by the
training data, which is a different claim and the one regulators ask about.

The clearest demonstration is to fit the same domain against both splits.

In [13]:
from qsarkit.applicability import KNNApplicabilityDomain, TanimotoSimilarityAD

for label, (tr, te) in [("random split", (rnd_train, rnd_test)),
                        ("scaffold split", (sca_train, sca_test))]:
    domain = KNNApplicabilityDomain(n_neighbors=3).fit(X[tr])
    inside = domain.predict(X[te])
    print(f"{label:16} {inside.sum()}/{len(te)} test compounds in domain")

random split     5/6 test compounds in domain
scaffold split   1/6 test compounds in domain


Nothing changed except which compounds were held out. A random split leaves
every test compound with a close analogue in training, so the domain looks
vacuous. A scaffold split holds out whole chemotypes, and the domain
correctly reports it has never seen anything like them.

**If your AD marks everything in-domain, suspect the split before
congratulating the model.**

In [14]:
from qsarkit.applicability import ADAnalyzer

# On a split with a mix of both, the domain earns its keep.
mixed_train = np.concatenate([sca_train[:14], sca_test[:2]])
mixed_test = np.array([i for i in range(len(Y)) if i not in set(mixed_train)])

model = QSARRegressor("rf", random_state=0).fit(X[mixed_train], Y[mixed_train])
analyzer = ADAnalyzer(TanimotoSimilarityAD(threshold=0.35)).fit(X[mixed_train])
ad_report = analyzer.report(X[mixed_test], Y[mixed_test], model.predict(X[mixed_test]))

for key, value in ad_report.items():
    print(f"{key:16} {value}")

coverage         0.75
n_inside         6
n_outside        2
rmse_inside      0.5712984115153991
rmse_outside     0.8404109113999132
mae_inside       0.5607166666666843
mae_outside      0.8345000000000193
rmse_ratio       1.4710541714454957


`rmse_ratio` is the number to read: greater than 1 means errors outside the
domain really are larger, so the domain separates reliable predictions from
unreliable ones. That is the evidence OECD principle 3 asks for.

## Uncertainty

An applicability domain answers "should I trust this at all". Uncertainty
answers "how wrong is it likely to be". Conformal prediction is the only
method here with a distribution-free coverage guarantee.

In [15]:
from qsarkit.uncertainty import ConformalRegressor

conformal = ConformalRegressor(
    QSARRegressor("rf", random_state=0), alpha=0.2, random_state=0
).fit(X[rnd_train], Y[rnd_train])

lower, upper = conformal.predict_interval(X[rnd_test])
evaluation = conformal.evaluate(X[rnd_test], Y[rnd_test])
print(f"target coverage {evaluation['expected_coverage']:.0%}, "
      f"achieved {evaluation['coverage']:.0%}")
print(f"mean interval width: {evaluation['mean_width']:.2f} log units")

target coverage 80%, achieved 100%
mean interval width: 4.29 log units


A 4-log-unit interval on data spanning 3.5 log units is the model being
honest that it knows very little from 18 training compounds. That is the
correct conclusion, and it is invisible in a point prediction.

In [16]:
from qsarkit.uncertainty import EnsembleUncertainty, UncertaintyCalibration

ensemble = EnsembleUncertainty(
    QSARRegressor("rf", random_state=0), random_state=0
).fit(X[rnd_train], Y[rnd_train])
mean, sigma = ensemble.predict_uncertainty(X[rnd_test])

calibration = UncertaintyCalibration(n_bins=3).report(Y[rnd_test], mean, sigma)
print(f"ENCE                       {calibration['ence']:.2f}   (0 is perfect)")
print(f"spearman(sigma, |error|)   {calibration['spearman_error_correlation']:+.2f}"
      "   (should be positive)")

ENCE                       2.57   (0 is perfect)
spearman(sigma, |error|)   -0.37   (should be positive)


Ensemble spread has no coverage guarantee — it is a proxy for uncertainty,
not a measurement of it. Check it before relying on it: a negative
correlation between predicted σ and actual error means the model is most
confident exactly where it is most wrong, which is worse than useless.

## The whole thing, as a pipe

Everything above, in the functional notation. Note that `split` comes before
`scale` and `select_features`, so nothing leaks.

In [17]:
from qsarkit.functional import (
    cross_validate, drop_invalid, fingerprint, fit, molecules,
    scale, select_features, split, standardize)

train, test = (
    molecules(SMILES, Y)
    >> standardize()
    >> drop_invalid()
    >> fingerprint("morgan", n_bits=512)
    >> split("scaffold", test_size=0.25)
)
model = train >> select_features(k=40) >> fit("rf", random_state=0)
print("train:", len(train), " test:", len(test))
print("fitted:", type(model).__name__)

train: 18  test: 6
fitted: QSARRegressor


In [18]:
report = (
    molecules(SMILES, Y)
    >> standardize()
    >> drop_invalid()
    >> fingerprint(n_bits=512)
    >> cross_validate("rf", n_splits=5, random_state=0)
)
print(f"Q2={report['q2']:.3f}  RMSE={report['rmse_cv']:.3f}")

Q2=0.694  RMSE=0.594


### Drawing the workflow

A pipeline is a graph, and drawing it is the quickest way to confirm the
stages are in the order you meant.

In [19]:
pipe = (standardize() >> drop_invalid() >> fingerprint(n_bits=512)
        >> scale() >> fit("rf"))
pipe.plot(title="QSAR workflow")

In [20]:
# Graphviz DOT source, for `dot -Tpdf`; or render directly:
#     pipe.render("workflow.pdf")
print(pipe.to_dot())

digraph qsarkit_pipeline {
  rankdir=TB;
  node [style="filled,rounded", shape=box, fontname="Helvetica", fontsize=11, margin="0.18,0.10"];
  edge [fontname="Helvetica", fontsize=9, color="#666666"];
  bgcolor="transparent";
  n0 [label="input\nmolecules + y", fillcolor="#E8EEF7", color="#41618F", shape=ellipse];
  n1 [label="standardize()", fillcolor="#DCEBDC", color="#3D7A3D", shape=box];
  n2 [label="drop_invalid()", fillcolor="#DCEBDC", color="#3D7A3D", shape=box];
  n3 [label="featurize(MorganFingerprint)", fillcolor="#FBEBD2", color="#B37A1E", shape=box];
  n4 [label="scale()", fillcolor="#E4E1F0", color="#5B4C93", shape=box];
  n5 [label="fit(estimator='rf')", fillcolor="#F7DEDE", color="#9E3B3B", shape=ellipse];
  n0 -> n1 [label=" MoleculeSet"];
  n1 -> n2 [label=" MoleculeSet"];
  n2 -> n3 [label=" MoleculeSet"];
  n3 -> n4 [label=" FeatureSet"];
  n4 -> n5 [label=" FeatureSet"];
}
